In [4]:
import OrcFxAPI
from pathlib import Path

owd_file = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\OW_data\Harlequin_mesh2_V3_owd.owd")
owr_file = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\OW_results\Mesh_DecayCalibration_V3_V1.owr")
xlsx_file = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\XLSX_results\Mesh_DecayCalibration_V3_V1.xlsx")


In [5]:
water_depth = 200.0  # m
wave_periods = [1, 2, 3, 4,5,6,7,8,9,10,11,12,13, 14, 15,16, 17, 18, 19,20]   # s
wave_headings = [0.0, 45.0, 90.0, 135.0, 180.0]
   # deg

mass = 65.61   # ton, alleen goed als jouw model ook ton verwacht
com_x = 0.0
com_y = 0.0
com_z = 10.03
# voorbeeld inertia matrix rond CoM
Ixx = 5061.21 
Iyy = 5061.21
Izz = 8346.92
Ixy = 0.0
Ixz = 0.0
Iyz = 0.0


# =========================
# HELPER FUNCTIONS
# =========================
def set_value(obj, name, value):
    """
    Zet een OrcaWave data item.
    De exacte data-itemnamen moet je uit OrcaWave halen met F7.
    """
    try:
        obj[name] = value
        print(f"Set: {name} = {value}")
    except Exception as e:
        print(f"Kon data item niet zetten: {name}")
        print(f"  Waarde: {value}")
        print(f"  Fout: {e}")


def print_validation(diff):
    info = getattr(diff, "ValidationInformationText", "")
    warnings = getattr(diff, "ValidationWarningText", "")
    errors = getattr(diff, "ValidationErrorText", "")

    print("\n=== VALIDATION INFO ===")
    print(info if info else "(geen info)")

    print("\n=== VALIDATION WARNINGS ===")
    print(warnings if warnings else "(geen warnings)")

    print("\n=== VALIDATION ERRORS ===")
    print(errors if errors else "(geen errors)")

    return errors


# =========================
# MAIN
# =========================
diff = OrcFxAPI.Diffraction()
diff.LoadData(str(owd_file))

# -------------------------------------------------
# BELANGRIJK:
# De namen hieronder zijn PLACEHOLDERS / typische structuur.
# Vervang ze met de exacte OrcaWave data names via F7 in de GUI.
# -------------------------------------------------

# Environment
diff.SetData("WaterDepth", 0, 200.0)

# Wave periods
# Vaak moet je eerst een count zetten en daarna de tabel vullen
diff.SetData("NumberOfPeriodsOrFrequencies", 0, len(wave_periods))
for i, T in enumerate(wave_periods):
    diff.SetData(f"PeriodOrFrequency", i, T)

# Wave headings
diff.SetData("NumberOfWaveHeadings", 0, len(wave_headings))
for i, hdg in enumerate(wave_headings):
    diff.SetData(f"WaveHeading", i, hdg)


diff.SetData("BodyInertiaSpecifiedBy", 0, "Matrix (for a general body)")
diff.SetData("BodyInertiaTensorOriginType", 0, "Centre of mass")
# Body inertia / mass properties
# Let op: body index / naam kan anders zijn in jouw model
diff.SetData("BodyCentreOfMassX", 0, com_x)
diff.SetData("BodyCentreOfMassY", 0, com_y)
diff.SetData("BodyCentreOfMassZ", 0, com_z)

diff.SetData("BodyMass", 0, mass)



diff.SetData("BodyInertiaTensorRx", 0, Ixx)   # xx
diff.SetData("BodyInertiaTensorRy", 1, Iyy)   # yy
diff.SetData("BodyInertiaTensorRz", 2, Izz)   # zz


print("Rx:", diff.BodyInertiaTensorRx[0])
print("Ry:", diff.BodyInertiaTensorRy[1])
print("Rz:", diff.BodyInertiaTensorRz[2])


# Optioneel: output settings
# Ook hier weer: exacte namen via F7
# set_value(diff, "CalculateDisplacementRAOs", "Yes")
# set_value(diff, "CalculateLoadRAOs", "Yes")
# set_value(diff, "CalculateAddedMassAndDamping", "Yes")
# set_value(diff, "CalculateWaveDriftQTFs", "No")

# Validation
errors = print_validation(diff)
if errors:
    raise RuntimeError("Model heeft validation errors. Fix eerst de data-itemnamen of invoer.")



Rx: 5061.21
Ry: 5061.21
Rz: 8346.92

=== VALIDATION INFO ===
('Estimated peak memory required during calculation: 108 MiB per thread.',)

=== VALIDATION WARNINGS ===
('Calculation mesh: the following panels are constructed from non-planar data in the mesh file: 2-4 6-8 19-23 35-40 50-56 67-72 83-88 99 101-104 114-120 130-132 134-136 147-152 162-168 178-184 194-197 199 200 210 211 213-216 226-230 232 242-248 258-260 262-264 275-279 291-296 306-312 323-328 339-344 355 357-360 370-376 386-388 390-392 404-408 418-424 434-440 450-453 455 456 467 469-472 482-486 488 498-504 514-520 530-534 536 547 549-552 562-565 567 568 578-584 594-600 612-616 626-628 630-632 642-648 659 661-664 675-680 691-696 706-712 723-728 739-743 754-756 758-760 770-776 786-790 792 802 803 805-808 818-821 823 824 834-840 850-856 867-872 882-884 886-888 898-904 915 917-920 931-936 947-952 962-968 979-984 995-999 1010-1012 1014-1016 1028-1032 1044-1048 1058-1064 1074-1080 1091-1096 1106-1112 1122 1124-1128 1140-1144 1156

In [6]:
# Run calculation
print("\n=== START CALCULATION ===")
diff.Calculate()
print("=== CALCULATION DONE ===")

# Save outputs
diff.SaveResults(str(owr_file))
diff.SaveResultsSpreadsheet(str(xlsx_file))

print("\nBestanden opgeslagen:")
print(f"  Results: {owr_file}")
print(f"  Spreadsheet: {xlsx_file}")


=== START CALCULATION ===
=== CALCULATION DONE ===

Bestanden opgeslagen:
  Results: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\OW_results\Mesh_DecayCalibration_V3_V1.owr
  Spreadsheet: C:\Users\verav\Desktop\Studie\Afstuderen\PHASE2_ORCA\XLSX_results\Mesh_DecayCalibration_V3_V1.xlsx
